# Stage 1: Setup and data

### 1.1 Environment

In [ ]:
pip install -q sentence-transformers datasets faiss-cpu indic-transliteration

In [ ]:
!pip -q install sentence-transformers pandas

In [ ]:
import sentence_transformers
import datasets
import faiss
import indic_transliteration

print("All libraries imported successfully!")

All libraries imported successfully!


## 1.2 Choose the base model by testing it

In [ ]:
import pandas as pd, unicodedata

rows = [
    (
        "2.47",
        (
            "कर्मण्येवाधिकारस्ते मा फलेषु कदाचन। मा"
            " कर्मफलहेतुर्भूर्मा ते सङ्गोऽस्त्वकर्मणि॥"
        ),
        (
            "karmany evadhikaras te ma phalesu kadacana | ma karma-phala-hetur"
            " bhur ma te sango 'stv akarmani ||"
        ),
        (
            "You have a right to perform your duty, but not to the fruits of"
            " your actions. Never be motivated by results, and never be"
            " attached to inaction."
        ),
    ),
    (
        "2.20",
        (
            "न जायते म्रियते वा कदाचिन्नायं भूत्वा भविता वा न भूयः। अजो नित्यः"
            " शाश्वतोऽयं पुराणो न हन्यते हन्यमाने शरीरे॥"
        ),
        (
            "na jayate mriyate va kadacin nayam bhutva bhavita va na bhuyah |"
            " ajo nityah sasvato 'yam purano na hanyate hanyamane sarire ||"
        ),
        (
            "The soul is never born and never dies. It is unborn, eternal and"
            " ancient, and it is not killed when the body is killed."
        ),
    ),
    (
        "2.22",
        (
            "वासांसि जीर्णानि यथा विहाय नवानि गृह्णाति नरोऽपराणि। तथा शरीराणि"
            " विहाय जीर्णान्यन्यानि संयाति नवानि देही॥"
        ),
        (
            "vasamsi jirnani yatha vihaya navani grhnati naro 'parani | tatha"
            " sarirani vihaya jirnany anyani samyati navani dehi ||"
        ),
        (
            "As a person discards worn-out clothes and puts on new ones, the"
            " soul discards worn-out bodies and enters new ones."
        ),
    ),
    (
        "4.7",
        (
            "यदा यदा हि धर्मस्य ग्लानिर्भवति भारत। अभ्युत्थानमधर्मस्य"
            " तदात्मानं सृजाम्यहम्॥"
        ),
        (
            "yada yada hi dharmasya glanir bhavati bharata | abhyutthanam"
            " adharmasya tadatmanam srjamy aham ||"
        ),
        (
            "Whenever righteousness declines and unrighteousness rises, I"
            " manifest myself."
        ),
    ),
    (
        "6.5",
        (
            "उद्धरेदात्मनात्मानं नात्मानमवसादयेत्। आत्मैव ह्यात्मनो बन्धुरात्मैव"
            " रिपुरात्मनः॥"
        ),
        (
            "uddhared atmanatmanam natmanam avasadayet | atmaiva hy atmano"
            " bandhur atmaiva ripur atmanah ||"
        ),
        (
            "Lift yourself by your own self and do not degrade yourself. The"
            " self alone is the friend of the self, and the self alone is its"
            " enemy."
        ),
    ),
    (
        "18.66",
        (
            "सर्वधर्मान्परित्यज्य मामेकं शरणं व्रज। अहं त्वां सर्वपापेभ्यो"
            " मोक्षयिष्यामि मा शुचः॥"
        ),
        (
            "sarva-dharman parityajya mam ekam saranam vraja | aham tvam"
            " sarva-papebhyo moksayisyami ma sucah ||"
        ),
        (
            "Abandon all varieties of duty and take refuge in Me alone. I will"
            " free you from all sins, so do not grieve."
        ),
    ),
]
df = pd.DataFrame(rows, columns=["id", "deva", "roman", "english"])
for c in ["deva", "roman", "english"]:
  df[c] = df[c].map(lambda s: unicodedata.normalize("NFC", s))
df

,id,deva,roman,english
0,2.47,कर्मण्येवाधिकारस्ते मा फलेषु कदाचन। मा कर्मफलह...,karmany evadhikaras te ma phalesu kadacana | m...,"You have a right to perform your duty, but not..."
1,2.20,न जायते म्रियते वा कदाचिन्नायं भूत्वा भविता वा...,na jayate mriyate va kadacin nayam bhutva bhav...,The soul is never born and never dies. It is u...
2,2.22,वासांसि जीर्णानि यथा विहाय नवानि गृह्णाति नरोऽ...,vasamsi jirnani yatha vihaya navani grhnati na...,As a person discards worn-out clothes and puts...
3,4.7,यदा यदा हि धर्मस्य ग्लानिर्भवति भारत। अभ्युत्थ...,yada yada hi dharmasya glanir bhavati bharata ...,Whenever righteousness declines and unrighteou...
4,6.5,उद्धरेदात्मनात्मानं नात्मानमवसादयेत्। आत्मैव ह...,uddhared atmanatmanam natmanam avasadayet | at...,Lift yourself by your own self and do not degr...
5,18.66,सर्वधर्मान्परित्यज्य मामेकं शरणं व्रज। अहं त्व...,sarva-dharman parityajya mam ekam saranam vraj...,Abandon all varieties of duty and take refuge ...


In [ ]:
from transformers import AutoTokenizer

models = {
  "e5-small-multi": "intfloat/multilingual-e5-small",
  "gte-multi-base": "Alibaba-NLP/gte-multilingual-base",
  "bge-small-en":   "BAAI/bge-small-en-v1.5",
}

def stats(tok, text):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    chars = len(text.replace(" ", ""))
    unk = sum(1 for i in ids if i == tok.unk_token_id)
    return len(ids), round(len(ids)/chars, 3), unk

out = []
for name, path in models.items():
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    for field in ["deva", "roman", "english"]:
        n, tpc, unk = zip(*[stats(tok, t) for t in df[field]])
        out.append(dict(model=name, script=field,
                        avg_tokens=sum(n)/len(n),
                        tokens_per_char=sum(tpc)/len(tpc),
                        unk_total=sum(unk)))
pd.DataFrame(out)

,model,script,avg_tokens,tokens_per_char,unk_total
0,e5-small-multi,deva,35.000000,0.440833,0
1,e5-small-multi,roman,38.333333,0.425000,0
2,e5-small-multi,english,30.833333,0.325333,0
3,gte-multi-base,deva,35.000000,0.440833,0
4,gte-multi-base,roman,38.333333,0.425000,0
5,gte-multi-base,english,30.833333,0.325333,0
6,bge-small-en,deva,54.166667,0.679000,9
7,bge-small-en,roman,40.166667,0.444500,0
8,bge-small-en,english,27.000000,0.281500,0


In [ ]:
for name, path in models.items():
    tok = AutoTokenizer.from_pretrained(path, trust_remote_code=True)
    print(name)
    print(" deva :", tok.tokenize(df.deva[0])[:25])
    print(" roman:", tok.tokenize(df.roman[0])[:25])

e5-small-multi
 deva : ['▁कर्म', 'ण्य', 'े', 'व', 'ाधिकार', 'स्ते', '▁मा', '▁फल', 'ेषु', '▁क', 'दा', 'चन', '।', '▁मा', '▁कर्म', 'फल', 'हे', 'त', 'ुर्', 'भू', 'र्', 'मा', '▁ते', '▁', 'सङ्ग']
 roman: ['▁karma', 'ṇ', 'y', '▁e', 'vā', 'dhi', 'kār', 'as', '▁te', '▁mā', '▁pha', 'le', 'ṣ', 'u', '▁kad', 'ā', 'cana', '▁', '|', '▁mā', '▁karma', '-', 'pha', 'la', '-']
gte-multi-base
 deva : ['▁कर्म', 'ण्य', 'े', 'व', 'ाधिकार', 'स्ते', '▁मा', '▁फल', 'ेषु', '▁क', 'दा', 'चन', '।', '▁मा', '▁कर्म', 'फल', 'हे', 'त', 'ुर्', 'भू', 'र्', 'मा', '▁ते', '▁', 'सङ्ग']
 roman: ['▁karma', 'ṇ', 'y', '▁e', 'vā', 'dhi', 'kār', 'as', '▁te', '▁mā', '▁pha', 'le', 'ṣ', 'u', '▁kad', 'ā', 'cana', '▁', '|', '▁mā', '▁karma', '-', 'pha', 'la', '-']
bge-small-en
 deva : ['क', '##र', '##म', '##ण', '##य', '##व', '##ा', '##ध', '##ि', '##क', '##ा', '##र', '##स', '##त', 'म', '##ा', '[UNK]', 'क', '##द', '##ा', '##च', '##न', '।', 'म', '##ा']
 roman: ['karma', '##ny', 'eva', '##dhi', '##kara', '##s', 'te', 'ma', 'ph', '##ales', '##u

In [ ]:
import numpy as np, torch
from sentence_transformers import SentenceTransformer

def prefixes(name):
    return ("query: ", "passage: ") if "e5" in name else ("", "")

def mrr_top1(sim):
    ranks = [(-sim[i]).argsort().tolist().index(i) + 1 for i in range(len(sim))]
    return np.mean([1/r for r in ranks]), np.mean([r == 1 for r in ranks])

tasks = {
  "deva -> english":  ("deva", "english"),
  "roman -> english": ("roman", "english"),
  "english -> deva":  ("english", "deva"),
  "deva -> roman":    ("deva", "roman"),
}

res = []
for name, path in models.items():
    try:
        m = SentenceTransformer(
            path, trust_remote_code=True, device="cpu",
            model_kwargs={"torch_dtype": torch.float32},
        )
        qp, dp = prefixes(name)
        for task, (qf, df_) in tasks.items():
            q = m.encode([qp + t for t in df[qf]], normalize_embeddings=True)
            d = m.encode([dp + t for t in df[df_]], normalize_embeddings=True)
            mrr, top1 = mrr_top1(q @ d.T)
            res.append(dict(model=name, task=task, MRR=round(mrr, 3), top1=round(top1, 3)))
        del m
    except Exception as e:
        print(f"FAILED {name}: {str(e)[:150]}")

pd.DataFrame(res).pivot(index="task", columns="model", values="MRR")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] NewModel LOAD REPORT from: Alibaba-NLP/gte-multilingual-base
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAILED gte-multi-base: index 133180126749344 is out of bounds for dimension 0 with size 46


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model,bge-small-en,e5-small-multi
task,,
deva -> english,0.389,0.806
deva -> roman,0.386,0.833
english -> deva,0.422,1.000
roman -> english,0.400,0.408


# Step 1.3: get the real Gita data

In [ ]:
from datasets import load_dataset
raw = load_dataset("OEvortex/Bhagavad_Gita", split="train").to_pandas()
print(raw.columns.tolist(), len(raw))
raw.head(3)

README.md:   0%|          | 0.00/1.14k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  288kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

['S.No.', 'Title', 'Chapter', 'Verse', 'Sanskrit Anuvad', 'Hindi Anuvad', 'Enlgish Translation'] 700


,S.No.,Title,Chapter,Verse,Sanskrit Anuvad,Hindi Anuvad,Enlgish Translation
0,1,Arjuna's Vishada Yoga,Chapter 1,Verse 1.1,धृतराष्ट्र उवाच । धर्मक्षेत्रे कुरुक्षेत्रे सम...,धृतराष्ट्र बोले- हे संजय! धर्मभूमि कुरुक्षेत्र...,"Dhrtarashtra asked of Sanjaya: O SANJAYA, what..."
1,2,Arjuna's Vishada Yoga,Chapter 1,Verse 1.2,सञ्जय उवाच । दृष्ट्वा तु पाण्डवानीकं व्यूढं दु...,संजय बोले- उस समय राजा दुर्योधन ने व्यूहरचनायु...,Sanjaya explained: Now seeing that the army of...
2,3,Arjuna's Vishada Yoga,Chapter 1,Verse 1.3,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् । व...,हे आचार्य! आपके बुद्धिमान्‌ शिष्य द्रुपदपुत्र ...,"Behold O, Master, the mighty army of the sons ..."


In [ ]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

raw["iast"] = raw["Sanskrit Anuvad"].map(
    lambda s: transliterate(s, sanscript.DEVANAGARI, sanscript.IAST))

In [ ]:
import re, unicodedata, pandas as pd
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

# 1. clean table with proper column names
df = pd.DataFrame({
    "id":      raw["Verse"].str.replace("Verse", "", regex=False).str.strip(),
    "chapter": raw["Chapter"].str.extract(r"(\d+)")[0].astype(int),
    "deva":    raw["Sanskrit Anuvad"].map(lambda s: unicodedata.normalize("NFC", s)),
    "english": raw["Enlgish Translation"],
})

# 2. Devanagari -> IAST -> ASCII
def to_ascii(s):
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace("'", "").replace("-", "").replace("|", " ")
    return re.sub(r"\s+", " ", s).strip()

df["iast"]  = df["deva"].map(lambda s: transliterate(s, sanscript.DEVANAGARI, sanscript.IAST))
df["ascii"] = df["iast"].map(to_ascii)

# 3. checks (these print)
print("rows:", len(df))
print("ids not in N.N format:", df.loc[~df.id.str.match(r"^\d+\.\d+$"), "id"].tolist())
print("rows with digits in Sanskrit:", df.deva.str.contains(r"[०-९0-9]").sum())
print("rows with speaker prefix (उवाच):", df.deva.str.contains("उवाच").sum())
print()
for c in ["deva", "iast", "ascii", "english"]:
    print(c.upper(), ":", df.loc[df.id == "2.47", c].values[0], "\n")

rows: 700
ids not in N.N format: []
rows with digits in Sanskrit: 600
rows with speaker prefix (उवाच): 33

DEVA : कर्मण्येवाधिकारस्ते मा फलेषु कदाचन । मा कर्मफलहेतुर्भूर्मा ते सङ्गोऽस्त्वकर्मणि ॥ २.४७ ॥ 

IAST : karmaṇyevādhikāraste mā phaleṣu kadācana | mā karmaphalaheturbhūrmā te saṅgo'stvakarmaṇi || 2.47 || 

ASCII : karmanyevadhikaraste ma phalesu kadacana ma karmaphalaheturbhurma te sangostvakarmani 2.47 

ENGLISH : O ARJUNA, always remember what I am about to say to you for it is the law of KARMA, a law that one should always obey in life should he/she ever feel resentment, frustration, anxiety, or grief:You have the right only to perform your actions, duties and responsibilities in life; however, the results of these actions should not concern you at all. You should not even desire results for your actions because the results are simply not in your hands, but in the hands of the Lord. Neither should you lean towards inaction. (This is the most important shloka describing Karmyog

In [ ]:
import re

NUM = r"[०-९0-9]+(?:\s*[.\-–]\s*[०-९0-9]+)*"

def clean_deva(s):
    s = re.sub(NUM, "", s)                        # remove verse numbers
    s = re.sub(r"॥\s*॥", "॥", s)                  # tidy leftover double danda
    # strip speaker prefix (≤3 words before the first danda)
    s = re.sub(r"^(?:\S+\s+){0,2}?\S*?(?:उवाच|ुवाच)\s*।\s*", "", s)
    return re.sub(r"\s+", " ", s).strip()

before = df["deva"].copy()
df["deva"]  = before.map(clean_deva)
df["iast"]  = df["deva"].map(lambda s: transliterate(s, sanscript.DEVANAGARI, sanscript.IAST))
df["ascii"] = df["iast"].map(to_ascii)

# checks
print("digits left in deva:", df.deva.str.contains(r"[०-९0-9]").sum())
print("digits left in ascii:", df["ascii"].str.contains(r"\d").sum())
print("speaker prefixes still present:", df.deva.str.contains("उवाच").sum())
print("rows changed:", (before != df.deva).sum())
print("empty deva:", (df.deva.str.len() == 0).sum())
print()
for i in ["2.47", "1.1", "10.1"]:
    r = df[df.id == i].iloc[0]
    print(i, "|", r.deva, "|", r["ascii"], "\n")

# length check (e5 limit = 512 tokens)
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")
for c in ["deva", "ascii", "english"]:
    L = df[c].map(lambda t: len(tok(t, add_special_tokens=False)["input_ids"]))
    print(c, "mean:", round(L.mean()), "max:", L.max(), "over 512:", (L > 512).sum())

digits left in deva: 0
digits left in ascii: 0
speaker prefixes still present: 3
rows changed: 634
empty deva: 0

2.47 | कर्मण्येवाधिकारस्ते मा फलेषु कदाचन । मा कर्मफलहेतुर्भूर्मा ते सङ्गोऽस्त्वकर्मणि ॥ | karmanyevadhikaraste ma phalesu kadacana ma karmaphalaheturbhurma te sangostvakarmani 

1.1 | धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः मामकाः पाण्डवाश्चैव किमकुर्वत सञ्जय ॥... | dharmaksetre kuruksetre samaveta yuyutsavah mamakah pandavascaiva kimakurvata sanjaya ... 

10.1 | भूय एव महाबाहो शृणु मे परमं वचः यत्तेऽहं प्रीयमाणाय वक्ष्यामि हितकाम्यया ॥ | bhuya eva mahabaho srnu me paramam vacah yatteham priyamanaya vaksyami hitakamyaya 

deva mean: 32 max: 43 over 512: 0
ascii mean: 29 max: 39 over 512: 0
english mean: 59 max: 167 over 512: 0


In [ ]:
df.head(5)

,id,chapter,deva,english,iast,ascii
0,1.1,1,धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः माम...,"Dhrtarashtra asked of Sanjaya: O SANJAYA, what...",dharmakṣetre kurukṣetre samavetā yuyutsavaḥ mā...,dharmaksetre kuruksetre samaveta yuyutsavah ma...
1,1.2,1,दृष्ट्वा तु पाण्डवानीकं व्यूढं दुर्योधनस्तदा आ...,Sanjaya explained: Now seeing that the army of...,dṛṣṭvā tu pāṇḍavānīkaṃ vyūḍhaṃ duryodhanastadā...,drstva tu pandavanikam vyudham duryodhanastada...
2,1.3,1,पश्यैतां पाण्डुपुत्राणामाचार्य महतीं चमूम् । व...,"Behold O, Master, the mighty army of the sons ...",paśyaitāṃ pāṇḍuputrāṇāmācārya mahatīṃ camūm | ...,pasyaitam panduputranamacarya mahatim camum vy...
3,1.4,1,अत्र शूरा महेष्वासा भीमार्जुनसमा युधि । युयुधा...,"Present here are the mighty archers, peers or ...",atra śūrā maheṣvāsā bhīmārjunasamā yudhi | yuy...,atra sura mahesvasa bhimarjunasama yudhi yuyud...
4,1.5,1,धृष्टकेतुश्चेकितानः काशिराजश्च वीर्यवान् । पुर...,"Dhrishtaketu, Chekitana, and the valiant king ...",dhṛṣṭaketuścekitānaḥ kāśirājaśca vīryavān | pu...,dhrstaketuscekitanah kasirajasca viryavan puru...


In [ ]:
df.to_csv("gita_clean.csv", index=False)
# columns: id, chapter, deva, iast, ascii, english

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
df.to_csv("/content/drive/MyDrive/gita_clean.csv", index=False)

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("gita_clean.csv")

train_ch, val_ch, test_ch = range(1, 14), [14, 15, 16], [17, 18]

train = df[df.chapter.isin(train_ch)].reset_index(drop=True)
val   = df[df.chapter.isin(val_ch)].reset_index(drop=True)
test  = df[df.chapter.isin(test_ch)].reset_index(drop=True)

# checks
print("sizes:", len(train), len(val), len(test), "| total:", len(train)+len(val)+len(test))
print("ids overlap:", len(set(train.id) & set(val.id)), len(set(train.id) & set(test.id)), len(set(val.id) & set(test.id)))

# exact-duplicate Sanskrit text across splits (leakage check)
tr = set(train.deva)
print("val verses also in train:", val.deva.isin(tr).sum())
print("test verses also in train:", test.deva.isin(tr).sum())

# save
train.to_csv("gita_train.csv", index=False)
val.to_csv("gita_val.csv", index=False)
test.to_csv("gita_test.csv", index=False)

sizes: 523 71 106 | total: 700
ids overlap: 0 0 0
val verses also in train: 0
test verses also in train: 0


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
dst = "/content/drive/MyDrive/sanskrit_retrieval/data"
os.makedirs(dst, exist_ok=True)

for f in ["gita_clean.csv", "gita_train.csv", "gita_val.csv", "gita_test.csv"]:
    shutil.copy(f, dst)

print(os.listdir(dst))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['gita_clean.csv', 'gita_train.csv', 'gita_val.csv', 'gita_test.csv']


# Stage 2: build training pairs and the eval set

2.1 Decide the query types

In [ ]:
df = pd.read_csv("gita_clean.csv", dtype={"id": str})
# after splitting, check:
print(train.id.is_unique, val.id.is_unique, test.id.is_unique)   # all True

False False False


In [ ]:
import pandas as pd
df = pd.read_csv("gita_clean.csv", dtype={"id": str})
print("unique:", df.id.is_unique, "| duplicates:", df.id.duplicated().sum())
print(df[df.id.duplicated(keep=False)].sort_values("id")[["id", "chapter"]].head(20))

unique: True | duplicates: 0
Empty DataFrame
Columns: [id, chapter]
Index: []


In [ ]:
import re, json, unicodedata, pandas as pd
from google.colab import drive
drive.mount("/content/drive")
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

D = "/content/drive/MyDrive/sanskrit_retrieval/data"
splits = {s: pd.read_csv(f"{D}/gita_{s}.csv", dtype={"id": str}) for s in ["train", "val", "test"]}

def to_ascii(s):
    s = unicodedata.normalize("NFD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.replace("'", "").replace("-", "").replace("|", " ")
    return re.sub(r"\s+", " ", s).strip()

def deva_to_iast(s):
    return transliterate(s, sanscript.DEVANAGARI, sanscript.IAST)

def half(deva):
    parts = [p.strip() for p in deva.replace("॥", "।").split("।") if p.strip()]
    return parts[0] if len(parts) >= 2 and len(parts[0].split()) >= 2 else None

RULES = [  # (qtype, query function, doc side)
 ("deva_full",       lambda r: r.deva,  "english"),
 ("iast_full",       lambda r: r.iast,  "english"),
 ("ascii_full",      lambda r: r.ascii, "english"),
 ("deva_half",       lambda r: half(r.deva), "english"),
 ("ascii_half",      lambda r: to_ascii(deva_to_iast(half(r.deva))) if half(r.deva) else None, "english"),
 ("ascii_short",     lambda r: " ".join(r.ascii.split()[:4]), "english"),
 ("ascii_to_deva",   lambda r: r.ascii, "deva"),
 ("english_to_deva", lambda r: r.english, "deva"),
]

def make_pairs(df, split):
    out = []
    for r in df.itertuples():
        for qtype, fn, side in RULES:
            q = fn(r)
            if q:
                out.append(dict(split=split, verse_id=r.id, qtype=qtype,
                                query=q, doc_side=side,
                                positive=r.english if side == "english" else r.deva))
    return out

for s, d in splits.items():
    pairs = make_pairs(d, s)
    with open(f"{D}/rule_pairs_{s}.jsonl", "w", encoding="utf-8") as f:
        for p in pairs:
            f.write(json.dumps(p, ensure_ascii=False) + "\n")
    print(s, len(pairs))
    print(pd.DataFrame(pairs).qtype.value_counts().to_string(), "\n")

# spot check: verse 2.47
for p in make_pairs(splits["train"][splits["train"].id == "2.47"], "train"):
    print(p["qtype"], "|", p["query"][:70], "->", p["doc_side"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train 4102
qtype
deva_full          523
iast_full          523
ascii_full         523
ascii_short        523
english_to_deva    523
ascii_to_deva      523
deva_half          482
ascii_half         482 

val 562
qtype
deva_full          71
iast_full          71
ascii_full         71
ascii_short        71
ascii_to_deva      71
english_to_deva    71
deva_half          68
ascii_half         68 

test 832
qtype
deva_full          106
iast_full          106
ascii_full         106
ascii_short        106
ascii_to_deva      106
english_to_deva    106
deva_half           98
ascii_half          98 

deva_full | कर्मण्येवाधिकारस्ते मा फलेषु कदाचन । मा कर्मफलहेतुर्भूर्मा ते सङ्गोऽस् -> english
iast_full | karmaṇyevādhikāraste mā phaleṣu kadācana | mā karmaphalaheturbhūrmā te -> english
ascii_full | karmanyevadhikaraste ma phalesu kadacana ma karmaphalaheturbhurma te s -> 

In [ ]:
train_ch, val_ch, test_ch = range(1, 14), [14, 15, 16], [17, 18]
train = df[df.chapter.isin(train_ch)].reset_index(drop=True)
val   = df[df.chapter.isin(val_ch)].reset_index(drop=True)
test  = df[df.chapter.isin(test_ch)].reset_index(drop=True)

print("sizes:", len(train), len(val), len(test), "| total:", len(train)+len(val)+len(test))
print("unique ids:", train.id.is_unique, val.id.is_unique, test.id.is_unique)   # all True
print("id overlap:", len(set(train.id)&set(val.id)), len(set(train.id)&set(test.id)), len(set(val.id)&set(test.id)))
print("val in train:", val.deva.isin(set(train.deva)).sum(), "| test in train:", test.deva.isin(set(train.deva)).sum())
print(train.id.head(12).tolist())   # should include 1.10, 1.11, not just 1.1

import shutil
for name, d in [("train", train), ("val", val), ("test", test)]:
    d.to_csv(f"gita_{name}.csv", index=False)
    shutil.copy(f"gita_{name}.csv", "/content/drive/MyDrive/sanskrit_retrieval/data/")

sizes: 523 71 106 | total: 700
unique ids: True True True
id overlap: 0 0 0
val in train: 0 | test in train: 0
['1.1', '1.2', '1.3', '1.4', '1.5', '1.6', '1.7', '1.8', '1.9', '1.10', '1.11', '1.12']


In [ ]:
import os; print([f for f in os.listdir(D) if f.startswith("rule_pairs")])

['rule_pairs_train.jsonl', 'rule_pairs_val.jsonl', 'rule_pairs_test.jsonl']


2.2 Build training pairs (train split only)

In [ ]:

print([m.name for m in client.models.list() if "flash" in m.name])

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview', 'models/gemini-omni-1.1-flash', 'models/gemini-3.6-flash', 'models/gemini-3.7-flash', 'models/gemini-3.8-flash', 'models/gemini-3.1-flash-tts-preview', 'models/gemini-2.5-flash-native-audio-latest', 'models/gemini-2.5-flash-native-audio-preview-09-2025', 'models/gemini-2.5-flash-native-audio-preview-12-2025', 'models/gemini-3.1-flash-live-preview']


In [ ]:
!pip -q install google-genai
import json, re, time
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

PROMPT = """You write search queries for a Bhagavad Gita retrieval system.
Given a verse's English translation, write 3 queries a real user might type to find it:
1. natural: a full question
2. keyword: 2-5 keywords, not a sentence
3. casual: informal, like a chat message

Rules:
- Paraphrase. Do NOT copy any phrase of 3+ words from the translation.
- Do not mention verse numbers, chapters, or "Bhagavad Gita".
- Ask about the idea or theme, not the wording.
Return only JSON: {"natural": "...", "keyword": "...", "casual": "..."}

Translation:
"""

MODEL = "gemini-3.1-flash-lite"

CFG = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.7,
    max_output_tokens=2048,   # headroom in case the model spends tokens on thinking
)

def gen(english):
    for attempt in range(4):
        try:
            r = client.models.generate_content(model=MODEL, contents=PROMPT + english, config=CFG)
            j = json.loads(r.text)
            if all(k in j for k in ["natural", "keyword", "casual"]):
                return j
        except Exception as e:
            msg = str(e)
            print("retry:", msg[:120])
            if "404" in msg or "NOT_FOUND" in msg or "400" in msg:
                raise           # bad model name or bad config: retrying won't help
            time.sleep(5 * (attempt + 1))
    return None

for r in splits["train"].head(5).itertuples():
    print(r.id, gen(r.english), "\n")

1.1 {'natural': 'What occurred when the two opposing armies assembled on the holy battlefield?', 'keyword': 'warrior factions gathering site', 'casual': 'tell me what happened at the start of the big battle'} 

1.2 {'natural': 'How did the opposing leader begin the confrontation after observing the enemy formation?', 'keyword': 'commander observation military strategy', 'casual': 'what did the prince tell his mentor when he saw the other side ready for battle?'} 

1.3 {'natural': 'How are the opposing military forces organized under their respective commanders?', 'keyword': 'battle formation military leadership', 'casual': 'who is leading the rival troops right now'} 

1.4 {'natural': 'Who were the primary military leaders aligned with the Pandava side during the conflict?', 'keyword': 'warrior commanders list battle', 'casual': 'tell me which powerful fighters were on the opposing team'} 

1.5 {'natural': 'Who were the notable warriors mentioned as allies in the opposing army?', 'keyw

In [ ]:
import os
DELAY = 0.5   # lower this if you have a paid key

for s, d in splits.items():
    path = f"{D}/llm_questions_{s}.jsonl"
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)["verse_id"] for l in open(path, encoding="utf-8")}
    with open(path, "a", encoding="utf-8") as f:
        for r in d.itertuples():
            if r.id in done:
                continue
            q = gen(r.english)
            if q is None:
                print("failed:", r.id); continue
            f.write(json.dumps({"split": s, "verse_id": r.id, **q}, ensure_ascii=False) + "\n")
            f.flush()
            time.sleep(DELAY)
    print(s, "done")

train done
val done
test done


In [ ]:
def ngrams(t, n=3):
    w = re.findall(r"\w+", t.lower()); return {tuple(w[i:i+n]) for i in range(len(w)-n+1)}

rows = []
for s in splits:
    eng = dict(zip(splits[s].id, splits[s].english))
    for l in open(f"{D}/llm_questions_{s}.jsonl", encoding="utf-8"):
        j = json.loads(l)
        for k in ["natural", "keyword", "casual"]:
            rows.append(dict(split=s, verse_id=j["verse_id"], qtype=f"q_{k}", query=j[k],
                             copied_3gram=len(ngrams(j[k]) & ngrams(eng[j["verse_id"]])) > 0,
                             has_number=bool(re.search(r"\d", j[k]))))
qdf = pd.DataFrame(rows)
print(qdf.groupby(["split", "qtype"]).size().unstack())
print("copied 3-grams:", qdf.copied_3gram.sum(), "| contains digits:", qdf.has_number.sum())
qdf.sample(20, random_state=0)[["verse_id", "qtype", "query"]]

qtype  q_casual  q_keyword  q_natural
split                                
test        106        106        106
train       523        523        523
val          71         71         71
copied 3-grams: 11 | contains digits: 0


,verse_id,qtype,query
2085,18.74,q_natural,How did the narrator react after sharing the d...
1941,18.26,q_natural,What are the characteristics of someone living...
1727,16.6,q_casual,tell me about the difference between kind and ...
1170,10.2,q_natural,Where does the divine essence reside within ev...
723,6.1,q_natural,What are the core characteristics and motivati...
1323,11.29,q_natural,Why does the vision show everyone being consum...
1055,9.15,q_casual,why are there so many ways to pray if god is j...
905,7.23,q_casual,is praying to minor spirits better than seekin...
1570,14.1,q_keyword,highest spiritual wisdom attainment
1825,17.15,q_keyword,"proper speech habits, truthful communication, ..."


In [ ]:
bad = qdf[qdf.copied_3gram | qdf.has_number]
print(bad[["verse_id", "qtype", "query"]].to_string())     # eyeball the 11

clean = qdf[~(qdf.copied_3gram | qdf.has_number)].drop(columns=["copied_3gram", "has_number"])
print(clean.groupby(["split", "qtype"]).size().unstack())

for s in ["train", "val", "test"]:
    clean[clean.split == s].to_json(f"{D}/llm_pairs_{s}.jsonl",
                                    orient="records", lines=True, force_ascii=False)

     verse_id      qtype                                                                                                        query
353      2.72   q_casual                                                     what happens to a person who finds inner peace for good?
438      3.29  q_natural  How should a spiritually enlightened individual interact with those who are blinded by material tendencies?
1572     14.2  q_natural              How does achieving spiritual enlightenment liberate a person from the cycle of birth and death?
1631    14.21   q_casual                                how do I spot someone who is no longer affected by the three modes of nature?
1754    16.15   q_casual                 what happens to people who are just super greedy and obsessed with showing off their wealth?
1920    18.19  q_natural                                    How do the three modes of nature influence our actions and understanding?
1934    18.23   q_casual                                      

2.3 Mine hard negatives

In [ ]:
import json, numpy as np, pandas as pd, torch
from sentence_transformers import SentenceTransformer

D = "/content/drive/MyDrive/sanskrit_retrieval/data"
train = pd.read_csv(f"{D}/gita_train.csv", dtype={"id": str})
ids  = train.id.tolist()
eng  = dict(zip(train.id, train.english))
deva = dict(zip(train.id, train.deva))
chap = dict(zip(train.id, train.chapter))
vnum = {i: int(i.split(".")[1]) for i in ids}
idx  = {i: k for k, i in enumerate(ids)}

def load(path): return [json.loads(l) for l in open(path, encoding="utf-8")]
pairs = load(f"{D}/rule_pairs_train.jsonl")
for p in load(f"{D}/llm_pairs_train.jsonl"):
    p["doc_side"], p["positive"] = "english", eng[p["verse_id"]]
    pairs.append(p)
print("total pairs:", len(pairs))

try:
    m = SentenceTransformer("intfloat/multilingual-e5-small", device="cuda")
    m.encode(["query: test"])          # forces a real GPU call to catch a broken context
except Exception as e:
    print("GPU failed, using CPU:", str(e)[:80])
    m = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
enc = lambda texts, pre: m.encode([pre + t for t in texts], normalize_embeddings=True,
                                  batch_size=64, show_progress_bar=False)

corpus = {"english": enc([eng[i] for i in ids], "passage: "),
          "deva":    enc([deva[i] for i in ids], "passage: ")}
text_of = {"english": eng, "deva": deva}
Q = enc([p["query"] for p in pairs], "query: ")

K_NEG, MARGIN = 5, 0.95
adjacent = lambda a, b: chap[a] == chap[b] and abs(vnum[a] - vnum[b]) <= 1

for p, q in zip(pairs, Q):
    sims = corpus[p["doc_side"]] @ q
    pk = idx[p["verse_id"]]
    order = np.argsort(-sims)
    p["pos_rank"] = int(np.where(order == pk)[0][0]) + 1
    negs = []
    for k in order:
        j = ids[k]
        if k == pk or adjacent(p["verse_id"], j) or sims[k] > MARGIN * sims[pk]:
            continue
        negs.append(j)
        if len(negs) == K_NEG:
            break
    p["neg_ids"] = negs
    p["negatives"] = [text_of[p["doc_side"]][j] for j in negs]

with open(f"{D}/train_pairs.jsonl", "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# checks
res = pd.DataFrame(pairs)
print(res.groupby("qtype").pos_rank.agg(
    top1=lambda r: (r == 1).mean(), top10=lambda r: (r <= 10).mean(), median="median").round(2))
print("negatives per pair:", res.neg_ids.map(len).value_counts().to_dict())

for _, r in res[res.qtype.isin(["q_natural", "ascii_full"])].sample(4, random_state=1).iterrows():
    print("\n", r.qtype, "|", r.query[:100])
    print(" POS:", r.positive[:120])
    for n in r.negatives[:3]:
        print(" NEG:", n[:120])

total pairs: 5669


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

GPU failed, using CPU: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https:/


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

                 top1  top10  median
qtype                               
ascii_full       0.01   0.06   169.0
ascii_half       0.02   0.08   177.0
ascii_short      0.03   0.09   182.0
ascii_to_deva    0.18   0.39    24.0
deva_full        0.10   0.28    48.0
deva_half        0.07   0.22    61.0
english_to_deva  0.16   0.43    14.0
iast_full        0.02   0.07   172.0
q_casual         0.11   0.32    27.0
q_keyword        0.12   0.40    20.0
q_natural        0.07   0.28    40.5
negatives per pair: {5: 4317, 0: 616, 1: 276, 2: 181, 3: 173, 4: 106}

 q_natural | What did the warrior observe when witnessing the supreme universal form?
 POS: O Lord, I have seen within Thy heavenly body, all of the gods and the infinite variety of being that have been created b
 NEG: That person who devotes himself to a life of KARMYOGA with no selfish motives in mind does not become hungry for power, 
 NEG: O Arjuna, knowing these two paths, no Yogi ever becomes confused, deluded or unhappy. Therefore Arjuna

In [ ]:
import json, random, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

D = "/content/drive/MyDrive/sanskrit_retrieval/data"
train = pd.read_csv(f"{D}/gita_train.csv", dtype={"id": str})
ids  = train.id.tolist()
eng  = dict(zip(train.id, train.english)); deva = dict(zip(train.id, train.deva))
chap = dict(zip(train.id, train.chapter)); vnum = {i: int(i.split(".")[1]) for i in ids}
text_of = {"english": eng, "deva": deva}

pairs = [json.loads(l) for l in open(f"{D}/train_pairs.jsonl", encoding="utf-8")]

m = SentenceTransformer("intfloat/multilingual-e5-small", device="cpu")
corpus = {s: m.encode(["passage: " + text_of[s][i] for i in ids], normalize_embeddings=True,
                      batch_size=64, show_progress_bar=False) for s in text_of}

POOL, K_NEG, CEIL = 15, 5, 0.95     # candidate pool, negatives per pair, near-duplicate ceiling
adjacent = lambda a, b: chap[a] == chap[b] and abs(vnum[a] - vnum[b]) <= 1

nbrs = {}
for s in corpus:
    S = corpus[s] @ corpus[s].T
    for i, vid in enumerate(ids):
        cand = [ids[k] for k in np.argsort(-S[i])
                if ids[k] != vid and not adjacent(vid, ids[k]) and S[i][k] < CEIL]
        nbrs[(s, vid)] = cand[:POOL]

rng = random.Random(0)
for p in pairs:
    pool = nbrs[(p["doc_side"], p["verse_id"])]
    p["neg_ids"] = rng.sample(pool, min(K_NEG, len(pool)))
    p["negatives"] = [text_of[p["doc_side"]][j] for j in p["neg_ids"]]

with open(f"{D}/train_pairs.jsonl", "w", encoding="utf-8") as f:
    for p in pairs:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

res = pd.DataFrame(pairs)
print("negatives per pair:", res.neg_ids.map(len).value_counts().to_dict())
for _, r in res[res.qtype.isin(["q_natural", "ascii_full"])].sample(4, random_state=1).iterrows():
    print("\n", r.qtype, "|", r.query[:100])
    print(" POS:", r.positive[:120])
    for n in r.negatives[:3]:
        print(" NEG:", n[:120])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

negatives per pair: {5: 5669}

 q_natural | What did the warrior observe when witnessing the supreme universal form?
 POS: O Lord, I have seen within Thy heavenly body, all of the gods and the infinite variety of being that have been created b
 NEG: O Lord, seeing Your enormous facial features and frightening teeth like the fire that burns till the end of time and all
 NEG: The Blessed lord said: Dear Arjuna, the Divine form of Myself which you have seen with so much difficulty is an experien
 NEG: O Lord, when I see Your vast and enormous form which reaches the sky, surrounded by a burning and magnificient glow of s

 ascii_full | durena hyavaram karma buddhiyogaddhanamjaya buddhau saranamanviccha krpanah phalahetavah
 POS: The Blessed Lord said unto ARJUNA:When actions are performed by a person for any selfish motive or gain, that person sha
 NEG: Lord Krishna continued: In my opinion, O Arjuna, without a controlled mind, it is difficult for one to attain Yoga (atta
 NEG: Arjuna aske

In [ ]:
!nvidia-smi
import torch
print(torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0))
x = torch.randn(4, 4, device="cuda"); print((x @ x).sum())

Sat Sep 19 09:38:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             31W /   70W |    3177MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


2.4 Build the eval set (val and test)

In [ ]:
import json, pandas as pd
D = "/content/drive/MyDrive/sanskrit_retrieval/data"
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]

for s in ["val", "test"]:
    rows = load(f"{D}/rule_pairs_{s}.jsonl")
    for p in load(f"{D}/llm_pairs_{s}.jsonl"):
        p["doc_side"] = "english"
        rows.append(p)
    ev = pd.DataFrame(rows)[["verse_id", "qtype", "query", "doc_side"]].rename(columns={"verse_id": "gold_id"})
    ev["ambiguous"] = False
    ev.to_json(f"{D}/{s}_eval.jsonl", orient="records", lines=True, force_ascii=False)

    g = pd.read_csv(f"{D}/gita_{s}.csv", dtype={"id": str})
    multi = ev.groupby(["query", "doc_side"]).gold_id.nunique()
    print(s, "| queries:", len(ev), "| verses:", len(g))
    print("  by type:", ev.qtype.value_counts().to_dict())
    print("  same query -> more than 1 gold verse:", (multi > 1).sum())
    print("  duplicate Sanskrit in split:", g.deva.duplicated().sum(), "| duplicate English:", g.english.duplicated().sum())

val | queries: 772 | verses: 71
  by type: {'deva_full': 71, 'iast_full': 71, 'ascii_full': 71, 'ascii_short': 71, 'ascii_to_deva': 71, 'english_to_deva': 71, 'q_keyword': 71, 'q_natural': 70, 'q_casual': 69, 'deva_half': 68, 'ascii_half': 68}
  same query -> more than 1 gold verse: 1
  duplicate Sanskrit in split: 0 | duplicate English: 1
test | queries: 1144 | verses: 106
  by type: {'deva_full': 106, 'iast_full': 106, 'ascii_full': 106, 'ascii_short': 106, 'ascii_to_deva': 106, 'english_to_deva': 106, 'q_keyword': 106, 'q_casual': 103, 'q_natural': 103, 'deva_half': 98, 'ascii_half': 98}
  same query -> more than 1 gold verse: 0
  duplicate Sanskrit in split: 0 | duplicate English: 0


In [ ]:
def review(s, n=25, seed=0):
    g = pd.read_csv(f"{D}/gita_{s}.csv", dtype={"id": str})
    eng = dict(zip(g.id, g.english))
    ev = pd.DataFrame(load(f"{D}/{s}_eval.jsonl"))
    for i, r in ev[ev.qtype.str.startswith("q_")].sample(n, random_state=seed).iterrows():
        print(f"[{i}] {r.qtype} | {r.query}\n     GOLD {r.gold_id}: {eng[r.gold_id][:160]}\n")

review("val")
review("test")

[574] q_keyword | three modes material nature bondage
     GOLD 14.5: Arjuna, that NATURE is made of three parts, namely: SATTVA (the light representing goodness); RAJAS (fire representing passion), and TAMAS (darkness representin

[642] q_keyword | inverted cosmic tree vedas symbolism
     GOLD 15.1: The Dear Lord explained to Arjuna:O Arjuna, there exists an enormous, everlasting, and divinely pure tree known as the indestructable ASVAATTHAM (COSMIC TREE OF

[595] q_keyword | signs of restless passion
     GOLD 14.12: When the RAJAS GUNA has taken over a being, then it can be seen through a person’s excessive greed, lust, unrest, activity and other similar action (therefore t

[567] q_natural | How does the origin of every living thing relate to the Divine?
     GOLD 14.3: O Arjuna, you must realize that it is from Me that all creation steam. The great Brahma, The Lord of all creation acts as My womb when I plant the seed of creat

[750] q_casual | is it bad to do good deeds just for

In [ ]:
DROP  = {"val": [], "test": []}    # e.g. [55, 130]  -> WRONG queries
AMBIG = {"val": [], "test": []}    # e.g. [12, 40]   -> AMBIGUOUS queries

for s in ["val", "test"]:
    ev = pd.DataFrame(load(f"{D}/{s}_eval.jsonl"))
    ev.loc[AMBIG[s], "ambiguous"] = True
    ev = ev.drop(index=DROP[s]).reset_index(drop=True)
    ev.to_json(f"{D}/{s}_eval.jsonl", orient="records", lines=True, force_ascii=False)
    print(s, len(ev), "queries |", int(ev.ambiguous.sum()), "flagged ambiguous")

val 772 queries | 0 flagged ambiguous
test 1144 queries | 0 flagged ambiguous


## Stage 3 preview: baseline and fine-tuning

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install -U sentence-transformers datasets accelerate

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 32.7 MB/s eta 0:00:00


In [ ]:
import json, pandas as pd, torch
D = "/content/drive/MyDrive/sanskrit_retrieval/data"
load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]

train_pairs = load(f"{D}/train_pairs.jsonl")
val_eval  = pd.DataFrame(load(f"{D}/val_eval.jsonl"))
test_eval = pd.DataFrame(load(f"{D}/test_eval.jsonl"))
val  = pd.read_csv(f"{D}/gita_val.csv",  dtype={"id": str})
test = pd.read_csv(f"{D}/gita_test.csv", dtype={"id": str})

In [ ]:
print(len(train_pairs), len(val_eval), len(test_eval))   # expect 5669, 772, 1144
print("cuda:", torch.cuda.is_available())
x = torch.randn(4, 4, device="cuda"); print((x @ x).sum())   # real GPU test

5669 772 1144
cuda: True


AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
